# Week 43 extension: zero-shot Telugu QA

**Hypothesis (stated before running).** The integrated QA model is fine-tuned on Arabic, Korean, and Telugu, while our zero-shot model is fine-tuned on Arabic and Korean only. We expect the zero-shot model to perform worse on Telugu because it has not been fine-tuned on Telugu QA examples.

- **Primary metrics:** Telugu Exact Match (EM) and token-level F1, compared with the integrated QA model on the same Telugu validation examples.
- **Expected direction:** zero-shot model < core model on both EM and token-level F1.
- **Falsified if:** the zero-shot model performs equally well or better on either primary metric. We will report the observed differences rather than assume the hypothesis is correct.
- **Secondary metric:** answerability macro-F1, to examine how well each model distinguishes answerable from unanswerable questions.

**Protocol:** Fine-tune `xlm-roberta-base` on Arabic and Korean training examples only. Use Arabic and Korean validation data for early stopping and model selection. Evaluate the selected model on the Telugu validation set, which is excluded from training and model selection. Compare it with the core integrated QA model using the same Telugu examples, answer-span evaluation procedure, and no-answer thresholding protocol. Report the scores and absolute differences. The challenge set is not used for this experiment.

In [ ]:
import os
import sys
import random
import numpy as np
import pandas as pd
import torch

from tqdm import tqdm
from pathlib import Path
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    get_linear_schedule_with_warmup,
)

from src.data import load_tokenised_data
from src.integrated_qa import (
    build_qa_features,
    QATrainDataset,
    train_collate_fn,
    val_collate_fn,
    f1_score_squad,
    exact_match_score,
)
from src.utils import enforce_reproducibility


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

enforce_reproducibility()

In [ ]:
df_train, df_val = load_tokenised_data()

train_source = df_train[df_train["lang"].isin(["ar", "ko"])].copy()
val_source = df_val[df_val["lang"].isin(["ar", "ko"])].copy()
test_telugu = df_val[df_val["lang"] == "te"].copy()

print("Arabic/Korean training examples:", len(train_source))
print("Arabic/Korean validation examples:", len(val_source))
print("Telugu evaluation examples:", len(test_telugu))
print("\nTraining language distribution:")
print(train_source["lang"].value_counts())

assert set(train_source["lang"].unique()) == {"ar", "ko"}
assert "te" not in set(val_source["lang"].unique())
assert "te" not in set(train_source["lang"].unique())
assert len(test_telugu) > 0

Arabic/Korean training examples: 4980
Arabic/Korean validation examples: 771
Telugu evaluation examples: 384

Training language distribution:
lang
ar    2558
ko    2422
Name: count, dtype: int64


In [ ]:
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 384
DOC_STRIDE = 128
BATCH_SIZE = 4
PREDICTION_BATCH_SIZE = 32

SAVE_PATH = Path("../checkpoints/zero_shot_telugu_qa")
CORE_PATH = Path("../checkpoints/integrated_qa")

# FIRST RUN: True trains and saves the zero-shot model.
# AFTER TRAINING: change this to False to load the saved checkpoint.
RUN_TRAINING = True

In [ ]:
if not RUN_TRAINING and not SAVE_PATH.exists():
    raise FileNotFoundError(
        f"Zero-shot checkpoint not found at {SAVE_PATH.resolve()}. "
        "Upload/copy the checkpoint folder to this location first."
    )

if RUN_TRAINING:
    # Start from the pretrained base model for the first training run.
    model = AutoModelForQuestionAnswering.from_pretrained(
        MODEL_NAME,
        attn_implementation="eager",
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
else:
    # Reuse the fine-tuned checkpoint on later runs.
    model = AutoModelForQuestionAnswering.from_pretrained(
        str(SAVE_PATH),
        attn_implementation="eager",
    )
    tokenizer = AutoTokenizer.from_pretrained(str(SAVE_PATH), use_fast=True)

model = model.to(device)
model.eval()

print("Model initialized.")
print("Training enabled:", RUN_TRAINING)
print("Model device:", next(model.parameters()).device)
print("Tokenizer loaded.")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForQuestionAnswering LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
qa_outputs.weight           | MISSING    | 
qa_outputs.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model initialized.
Training enabled: True
Model device: cuda:0
Tokenizer loaded.


In [ ]:
val_features = build_qa_features(
    val_source,
    tokenizer,
    max_length=MAX_LENGTH,
    doc_stride=DOC_STRIDE,
)

telugu_features = build_qa_features(
    test_telugu,
    tokenizer,
    max_length=MAX_LENGTH,
    doc_stride=DOC_STRIDE,
)

print("Arabic/Korean validation feature windows:", len(val_features))
print("Telugu feature windows:", len(telugu_features))

if RUN_TRAINING:
    train_features = build_qa_features(
        train_source,
        tokenizer,
        max_length=MAX_LENGTH,
        doc_stride=DOC_STRIDE,
    )
    print("Training feature windows:", len(train_features))

Arabic/Korean validation feature windows: 796
Telugu feature windows: 390
Training feature windows: 5144


In [ ]:
LR = 2e-5
N_EPOCHS = 20
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 200
PATIENCE = 3

if RUN_TRAINING:
    # Freeze the encoder first, then unfreeze the final four transformer layers.
    for param in model.roberta.parameters():
        param.requires_grad = False

    for layer in model.roberta.encoder.layer[-4:]:
        for param in layer.parameters():
            param.requires_grad = True

    for param in model.qa_outputs.parameters():
        param.requires_grad = True

    train_dataset = QATrainDataset(train_features)
    train_dl = DataLoader(
        train_dataset,
        collate_fn=train_collate_fn,
        shuffle=True,
        batch_size=BATCH_SIZE,
    )

    def qa_loss_collate_fn(batch):
        keys = ["input_ids", "attention_mask", "start_positions", "end_positions"]
        return {
            key: torch.tensor([item[key] for item in batch], dtype=torch.long)
            for key in keys
        }

    val_loss_dl = DataLoader(
        val_features,
        collate_fn=qa_loss_collate_fn,
        batch_size=32,
        shuffle=False,
    )

    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = [
        {
            "params": [
                p for name, p in model.named_parameters()
                if p.requires_grad and not any(nd in name for nd in no_decay)
            ],
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": [
                p for name, p in model.named_parameters()
                if p.requires_grad and any(nd in name for nd in no_decay)
            ],
            "weight_decay": 0.0,
        },
    ]
    optimizer = AdamW(optimizer_grouped_parameters, lr=LR)
    total_training_steps = N_EPOCHS * len(train_dl)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=WARMUP_STEPS,
        num_training_steps=total_training_steps,
    )

    def evaluate_validation_loss(model, data_loader, device):
        model.eval()
        total_loss = 0.0
        total_windows = 0
        with torch.inference_mode():
            for batch in tqdm(data_loader, desc="Validation loss", leave=False):
                batch = {key: value.to(device) for key, value in batch.items()}
                outputs = model(**batch)
                n = batch["input_ids"].shape[0]
                total_loss += outputs.loss.item() * n
                total_windows += n
        return total_loss / max(total_windows, 1)

    def train_model(model, train_dl, val_loss_dl, optimizer, scheduler,
                    n_epochs, device, checkpoint_path, patience=3):
        history = []
        best_val_loss = float("inf")
        best_epoch = 0
        epochs_without_improvement = 0
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

        for epoch in range(n_epochs):
            model.train()
            train_losses = []
            for batch in tqdm(train_dl, desc=f"Epoch {epoch + 1}/{n_epochs}"):
                batch = {key: value.to(device) for key, value in batch.items()}
                optimizer.zero_grad()
                outputs = model(**batch)
                loss = outputs.loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                train_losses.append(loss.item())

            train_loss = float(np.mean(train_losses))
            val_loss = evaluate_validation_loss(model, val_loss_dl, device)
            history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})
            print(f"Epoch {epoch + 1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_epoch = epoch + 1
                epochs_without_improvement = 0
                torch.save(model.state_dict(), checkpoint_path)
                model.save_pretrained(str(SAVE_PATH))
                tokenizer.save_pretrained(str(SAVE_PATH))
                print("Saved best checkpoint.")
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= patience:
                    print("Early stopping.")
                    break

        model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
        model.eval()
        print("Best epoch:", best_epoch)
        print("Best validation loss:", best_val_loss)
        return pd.DataFrame(history)

    checkpoint_path = SAVE_PATH / "best_state_dict.pt"
    history = train_model(
        model=model,
        train_dl=train_dl,
        val_loss_dl=val_loss_dl,
        optimizer=optimizer,
        scheduler=scheduler,
        n_epochs=N_EPOCHS,
        device=device,
        checkpoint_path=checkpoint_path,
        patience=PATIENCE,
    )
    history.to_csv(SAVE_PATH / "training_history.csv", index=False)
    print("Saved model and training history to:", SAVE_PATH.resolve())
else:
    print("Skipping training: loading the saved zero-shot checkpoint.")

Epoch 1/20: 100%|██████████| 1286/1286 [02:34<00:00,  8.31it/s]


Epoch 1: train_loss=3.0796, val_loss=2.4393


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best checkpoint.


Epoch 2/20: 100%|██████████| 1286/1286 [02:34<00:00,  8.30it/s]


...

Epoch 4: train_loss=1.4296, val_loss=1.5959


Epoch 5/20: 100%|██████████| 1286/1286 [02:29<00:00,  8.62it/s]


Epoch 5: train_loss=1.2544, val_loss=1.6746


Epoch 6/20: 100%|██████████| 1286/1286 [02:28<00:00,  8.64it/s]


Epoch 6: train_loss=1.0600, val_loss=1.7404
Early stopping.
Best epoch: 3
Best validation loss: 1.592526503543758
Saved model and training history to: /content/nlp-ku-project/checkpoints/zero_shot_telugu_qa


In [ ]:
def prepare_prediction_features(features):
    return [
        {
            "input_ids": feature["input_ids"],
            "attention_mask": feature["attention_mask"],
            "example_index": feature["example_index"],
            "offset_mapping": [
                offset if seq_id == 1 else None
                for offset, seq_id in zip(
                    feature["offset_mapping"],
                    feature["sequence_ids"],
                )
            ],
        }
        for feature in features
    ]

In [ ]:
def predict(model, data_loader, device, description="Predicting"):
    model.eval()
    start_logits_all = []
    end_logits_all = []
    with torch.inference_mode():
        for batch in tqdm(data_loader, desc=description):
            batch = {key: value.to(device) for key, value in batch.items()}
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )
            start_logits_all.extend(outputs.start_logits.cpu().numpy())
            end_logits_all.extend(outputs.end_logits.cpu().numpy())
    return start_logits_all, end_logits_all

In [ ]:
def post_process_predictions(
    examples,
    features,
    logits,
    threshold=np.inf,
    num_possible_answers=20,
    max_answer_length=30,
):
    start_logits_all, end_logits_all = logits
    features_per_example = defaultdict(list)
    for feature_index, feature in enumerate(features):
        features_per_example[int(feature["example_index"])].append(feature_index)

    rows = []
    examples_reset = examples.reset_index(drop=True)
    for example_idx, sample in examples_reset.iterrows():
        feature_indices = features_per_example.get(example_idx, [])
        context = str(sample["context"])
        best_span_score = -np.inf
        best_span_text = ""
        best_null_score = np.inf

        for feature_idx in feature_indices:
            start_logits = start_logits_all[feature_idx]
            end_logits = end_logits_all[feature_idx]
            offsets = features[feature_idx]["offset_mapping"]
            null_score = float(start_logits[0] + end_logits[0])
            best_null_score = min(best_null_score, null_score)
            start_indices = np.argsort(start_logits)[::-1][:num_possible_answers]
            end_indices = np.argsort(end_logits)[::-1][:num_possible_answers]

            for start_index in start_indices:
                for end_index in end_indices:
                    if start_index >= len(offsets) or end_index >= len(offsets):
                        continue
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    if start_index > end_index:
                        continue
                    if end_index - start_index + 1 > max_answer_length:
                        continue
                    start_char = offsets[start_index][0]
                    end_char = offsets[end_index][1]
                    if end_char <= start_char:
                        continue
                    score = float(start_logits[start_index] + end_logits[end_index])
                    if score > best_span_score:
                        best_span_score = score
                        best_span_text = context[start_char:end_char]

        # If no valid span exists, avoid a misleading infinite score.
        if best_span_score == -np.inf:
            no_answer_score = np.inf
            best_span_text = ""
        else:
            no_answer_score = best_null_score - best_span_score

        prediction = "" if no_answer_score > threshold else best_span_text
        rows.append({
            "example_index": example_idx,
            "prediction": prediction,
            "no_answer_score": float(no_answer_score),
        })
    return pd.DataFrame(rows)

In [ ]:
def add_gold_labels(predictions, examples):
    examples_reset = examples.reset_index(drop=True)
    predictions = predictions.copy()
    predictions["gold_answer"] = [
        str(examples_reset.loc[i, "answer"])
        if pd.notna(examples_reset.loc[i, "answer"]) and bool(examples_reset.loc[i, "answerable"])
        else ""
        for i in predictions["example_index"]
    ]
    predictions["gold_answerable"] = [
        int(bool(examples_reset.loc[i, "answerable"]))
        for i in predictions["example_index"]
    ]
    predictions["lang"] = [
        examples_reset.loc[i, "lang"] for i in predictions["example_index"]
    ]
    return predictions

In [ ]:
def select_threshold(validation_predictions, n_thresholds=101):
    scores = validation_predictions["no_answer_score"].replace(
        [np.inf, -np.inf], np.nan
    ).dropna()
    if scores.empty:
        raise ValueError("No finite no-answer scores available for threshold selection.")
    thresholds = np.linspace(scores.min(), scores.max(), n_thresholds)
    results = []
    for threshold in thresholds:
        preds = np.where(
            validation_predictions["no_answer_score"] > threshold,
            "",
            validation_predictions["prediction"],
        )
        em = np.mean([
            exact_match_score(pred, gold)
            for pred, gold in zip(preds, validation_predictions["gold_answer"])
        ])
        token_f1 = np.mean([
            f1_score_squad(pred, gold)
            for pred, gold in zip(preds, validation_predictions["gold_answer"])
        ])
        results.append({"threshold": float(threshold), "EM": float(em), "Token F1": float(token_f1)})
    results_df = pd.DataFrame(results)
    best_row = results_df.loc[results_df["Token F1"].idxmax()]
    return float(best_row["threshold"]), results_df, best_row

In [ ]:
def calculate_metrics(predictions, threshold):
    result = predictions.copy()
    result["prediction_tuned"] = np.where(
        result["no_answer_score"] > threshold,
        "",
        result["prediction"],
    )
    result["EM"] = [
        exact_match_score(pred, gold)
        for pred, gold in zip(result["prediction_tuned"], result["gold_answer"])
    ]
    result["Token F1"] = [
        f1_score_squad(pred, gold)
        for pred, gold in zip(result["prediction_tuned"], result["gold_answer"])
    ]
    result["pred_answerable"] = (result["prediction_tuned"].str.len() > 0).astype(int)
    macro_f1 = f1_score(
        result["gold_answerable"],
        result["pred_answerable"],
        average="macro",
        zero_division=0,
    )
    metrics = {
        "EM": float(result["EM"].mean()),
        "Token F1": float(result["Token F1"].mean()),
        "Answerability macro-F1": float(macro_f1),
    }
    return result, metrics

In [ ]:
from collections import defaultdict
source_prediction_features = prepare_prediction_features(val_features)
source_dl = DataLoader(
    source_prediction_features,
    collate_fn=val_collate_fn,
    batch_size=PREDICTION_BATCH_SIZE,
    shuffle=False,
)

print("Arabic/Korean validation examples:", len(val_source))
print("Validation feature windows:", len(source_prediction_features))
print("Validation batches:", len(source_dl))

source_logits = predict(model, source_dl, device, description="Arabic/Korean validation")
source_predictions = post_process_predictions(
    val_source,
    source_prediction_features,
    source_logits,
    threshold=np.inf,
)
source_predictions = add_gold_labels(source_predictions, val_source)

print("Validation predictions:", len(source_predictions))
display(source_predictions.head())

Arabic/Korean validation examples: 771
Validation feature windows: 796
Validation batches: 25


Arabic/Korean validation: 100%|██████████| 25/25 [00:18<00:00,  1.38it/s]


Validation predictions: 771


,example_index,prediction,no_answer_score,gold_answer,gold_answerable,lang
0,0,Royal Academy of Åbo,-11.465612,Royal Academy of Åbo,1,ar
1,1,"Finland, Sweden, Denmark, Estonia, Latvia, Lit...",-17.539681,"Finland, Sweden, Denmark, Estonia, Latvia, Lit...",1,ar
2,2,Grantham,-6.617302,Grantham,1,ar
3,3,"1328 or 1330, he made his way to the port of J...",9.796698,,0,ar
4,4,Abdrabbuh Mansur Hadi,-18.423708,Ali Abdullah Saleh,1,ar


In [ ]:
best_threshold, threshold_results, best_row = select_threshold(source_predictions)

print("Selected threshold:", best_threshold)
print("Arabic/Korean validation EM at selected threshold:", float(best_row["EM"]))
print("Arabic/Korean validation token F1 at selected threshold:", float(best_row["Token F1"]))

display(threshold_results.sort_values("Token F1", ascending=False).head())

Selected threshold: 0.13069893360137996
Arabic/Korean validation EM at selected threshold: 0.4993514915693904
Arabic/Korean validation token F1 at selected threshold: 0.6004060386520796


,threshold,EM,Token F1
72,1.161065,0.499351,0.600406
69,0.130699,0.499351,0.600406
70,0.474154,0.499351,0.600406
71,0.817610,0.499351,0.600406
68,-0.212757,0.498054,0.599109


In [ ]:
from sklearn.metrics import f1_score

telugu_prediction_features = prepare_prediction_features(telugu_features)
telugu_dl = DataLoader(
    telugu_prediction_features,
    collate_fn=val_collate_fn,
    batch_size=PREDICTION_BATCH_SIZE,
    shuffle=False,
)

print("Telugu examples:", len(test_telugu))
print("Telugu feature windows:", len(telugu_prediction_features))
print("Telugu prediction batches:", len(telugu_dl))

telugu_logits = predict(model, telugu_dl, device, description="Telugu evaluation")
telugu_predictions = post_process_predictions(
    test_telugu,
    telugu_prediction_features,
    telugu_logits,
    threshold=np.inf,
)
telugu_predictions = add_gold_labels(telugu_predictions, test_telugu)
zero_shot_predictions, zero_shot_metrics = calculate_metrics(
    telugu_predictions,
    best_threshold,
)

print("Zero-shot Telugu metrics:")
for metric, value in zero_shot_metrics.items():
    print(f"{metric}: {value:.4f}")

display(zero_shot_predictions.head())

Telugu examples: 384
Telugu feature windows: 390
Telugu prediction batches: 13


Telugu evaluation: 100%|██████████| 13/13 [00:07<00:00,  1.68it/s]


Zero-shot Telugu metrics:
EM: 0.3568
Token F1: 0.4327
Answerability macro-F1: 0.4927


,example_index,prediction,no_answer_score,gold_answer,gold_answerable,lang,prediction_tuned,EM,Token F1,pred_answerable
0,0,Portland,-15.503311,Portland,1,te,Portland,True,1.0,1
1,1,Indonesia,-4.118499,Indian subcontinent,1,te,Indonesia,False,0.0,1
2,2,Russia,-4.450328,England,1,te,Russia,False,0.0,1
3,3,1914 to 1918,-11.320415,1914,1,te,1914 to 1918,False,0.5,1
4,4,28 July 1914,-13.112836,28 July 1914,1,te,28 July 1914,True,1.0,1


In [ ]:
comparison_rows = [{"Model": "Zero-shot Arabic/Korean", **zero_shot_metrics, "Threshold": best_threshold}]

if CORE_PATH.exists() and (CORE_PATH / "config.json").exists():
    print("Loading core model from:", CORE_PATH.resolve())
    core_model = AutoModelForQuestionAnswering.from_pretrained(
        str(CORE_PATH),
        attn_implementation="eager",
    ).to(device)
    core_tokenizer = AutoTokenizer.from_pretrained(str(CORE_PATH), use_fast=True)
    core_model.eval()

    core_val_features = build_qa_features(
        val_source,
        core_tokenizer,
        max_length=MAX_LENGTH,
        doc_stride=DOC_STRIDE,
    )
    core_telugu_features = build_qa_features(
        test_telugu,
        core_tokenizer,
        max_length=MAX_LENGTH,
        doc_stride=DOC_STRIDE,
    )

    core_source_pf = prepare_prediction_features(core_val_features)
    core_source_dl = DataLoader(
        core_source_pf,
        collate_fn=val_collate_fn,
        batch_size=PREDICTION_BATCH_SIZE,
        shuffle=False,
    )
    core_source_logits = predict(
        core_model, core_source_dl, device,
        description="Core model Arabic/Korean validation",
    )
    core_source_predictions = post_process_predictions(
        val_source, core_source_pf, core_source_logits, threshold=np.inf
    )
    core_source_predictions = add_gold_labels(core_source_predictions, val_source)
    core_threshold, core_threshold_results, core_best_row = select_threshold(core_source_predictions)

    core_telugu_pf = prepare_prediction_features(core_telugu_features)
    core_telugu_dl = DataLoader(
        core_telugu_pf,
        collate_fn=val_collate_fn,
        batch_size=PREDICTION_BATCH_SIZE,
        shuffle=False,
    )
    core_telugu_logits = predict(
        core_model, core_telugu_dl, device,
        description="Core model Telugu evaluation",
    )
    core_telugu_predictions = post_process_predictions(
        test_telugu, core_telugu_pf, core_telugu_logits, threshold=np.inf
    )
    core_telugu_predictions = add_gold_labels(core_telugu_predictions, test_telugu)
    core_predictions_tuned, core_metrics = calculate_metrics(
        core_telugu_predictions, core_threshold
    )
    comparison_rows.append({
        "Model": "Core integrated QA",
        **core_metrics,
        "Threshold": core_threshold,
    })
    print("Core model threshold selected on Arabic/Korean validation:", core_threshold)
else:
    print(
        "Core checkpoint not found at", CORE_PATH.resolve(),
        "— comparison skipped. Make sure the integrated_qa checkpoint is available to complete the comparison."
    )

comparison = pd.DataFrame(comparison_rows)
display(comparison)

if len(comparison) == 2:
    core_row = comparison.loc[comparison["Model"] == "Core integrated QA"].iloc[0]
    zero_row = comparison.loc[comparison["Model"] == "Zero-shot Arabic/Korean"].iloc[0]
    print("\nZero-shot minus core (negative means lower zero-shot score):")
    for metric in ["EM", "Token F1", "Answerability macro-F1"]:
        print(f"{metric}: {zero_row[metric] - core_row[metric]:+.4f}")

Loading core model from: /content/nlp-ku-project/checkpoints/integrated_qa


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Core model Telugu evaluation: 100%|██████████| 13/13 [00:07<00:00,  1.71it/s]


Core model threshold selected on Arabic/Korean validation: 0.7957053422927842


,Model,EM,Token F1,Answerability macro-F1,Threshold
0,Zero-shot Arabic/Korean,0.356771,0.432699,0.492689,0.130699
1,Core integrated QA,0.348958,0.432735,0.513867,0.795705



Zero-shot minus core (negative means lower zero-shot score):
EM: +0.0078
Token F1: -0.0000
Answerability macro-F1: -0.0212


In [ ]:
# 1. Selected threshold
print("1. NO-ANSWER THRESHOLD")
print(f"Selected using Arabic/Korean validation data: {best_threshold:.4f}")

selected_row = threshold_results.loc[
    (threshold_results["threshold"] - best_threshold).abs().idxmin()
]

display(
    selected_row[["threshold", "EM", "Token F1"]]
    .to_frame()
    .T
    .round(4)
)

# 2. Core model vs zero-shot extension
print("\n2. MODEL COMPARISON ON TELUGU")

display(comparison.round(4))

# 3. A small sample of predictions
print("\n3. EXAMPLE TELUGU PREDICTIONS")

prediction_columns = [
    "example_index",
    "prediction_tuned",
    "gold_answer",
    "gold_answerable",
    "EM",
    "Token F1",
    "no_answer_score",
]

display(zero_shot_predictions[prediction_columns].head(8))

# 4. Examples where the zero-shot model made mistakes
print("\n4. EXAMPLES OF INCORRECT PREDICTIONS")

incorrect = zero_shot_predictions[
    ~zero_shot_predictions["EM"].astype(bool)
]

display(incorrect[prediction_columns].head(8))

print(f"\nTotal Telugu examples: {len(zero_shot_predictions)}")
print(f"Incorrect exact-match predictions: {len(incorrect)}")

1. NO-ANSWER THRESHOLD
Selected using Arabic/Korean validation data: 0.1307


,threshold,EM,Token F1
69,0.1307,0.4994,0.6004



2. MODEL COMPARISON ON TELUGU


,Model,EM,Token F1,Answerability macro-F1,Threshold
0,Zero-shot Arabic/Korean,0.3568,0.4327,0.4927,0.1307
1,Core integrated QA,0.3490,0.4327,0.5139,0.7957



3. EXAMPLE TELUGU PREDICTIONS


,example_index,prediction_tuned,gold_answer,gold_answerable,EM,Token F1,no_answer_score
0,0,Portland,Portland,1,True,1.0,-15.503311
1,1,Indonesia,Indian subcontinent,1,False,0.0,-4.118499
2,2,Russia,England,1,False,0.0,-4.450328
3,3,1914 to 1918,1914,1,False,0.5,-11.320415
4,4,28 July 1914,28 July 1914,1,True,1.0,-13.112836
5,5,11 May 2005,India,1,False,0.0,-8.619508
6,6,122,122,1,True,1.0,-15.278023
7,7,5 liters,approximately 5 liters,1,False,0.8,-16.653772



4. EXAMPLES OF INCORRECT PREDICTIONS


,example_index,prediction_tuned,gold_answer,gold_answerable,EM,Token F1,no_answer_score
1,1,Indonesia,Indian subcontinent,1,False,0.000000,-4.118499
2,2,Russia,England,1,False,0.000000,-4.450328
3,3,1914 to 1918,1914,1,False,0.500000,-11.320415
5,5,11 May 2005,India,1,False,0.000000,-8.619508
7,7,5 liters,approximately 5 liters,1,False,0.800000,-16.653772
9,9,21 years to 18 years,18,1,False,0.333333,-8.266471
11,11,India has the largest paddy output in the worl...,West Bengal,1,False,0.173913,-13.298381
12,12,122,"122 which were spoken by more than 10,000 people",1,False,0.200000,-15.814200



Total Telugu examples: 384
Incorrect exact-match predictions: 247


## Results and Discussion

The zero-shot model achieved an Exact Match score of 0.3568 and a token-level F1 score of 0.4327 on Telugu. The core integrated QA model achieved an Exact Match score of 0.3490 and a token-level F1 score of 0.4327.

Our hypothesis was that the zero-shot model would perform worse than the core model on both metrics. However, the results did not support this expectation: the zero-shot model had a slightly higher Exact Match score, while the token-level F1 scores were almost identical. Its answerability macro-F1 was lower (0.4927 compared with 0.5139), suggesting that distinguishing answerable from unanswerable questions remains a challenge.

These results suggest that fine-tuning on Telugu did not lead to a clear improvement in answer extraction in this experiment.

In [35]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path("/content/nlp-ku-project")

assert PROJECT_ROOT.exists(), (
    f"Project not found at {PROJECT_ROOT}. "
    "Update PROJECT_ROOT to your actual repository location."
)

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ZERO_SHOT_PATH = PROJECT_ROOT / "checkpoints" / "zero_shot_telugu_qa"
CHALLENGE_PATH = PROJECT_ROOT / "student_starter" / "challenge.json"

required_files = [
    PROJECT_ROOT / "src" / "challenge_eval.py",
    PROJECT_ROOT / "student_starter" / "validate_challenge.py",
    CHALLENGE_PATH,
]

print("Project root:", PROJECT_ROOT)

for path in required_files:
    print("OK" if path.exists() else "MISSING", path)

print("\nZero-shot checkpoint:", ZERO_SHOT_PATH)
if ZERO_SHOT_PATH.exists():
    print("Checkpoint files:", [
        p.name for p in ZERO_SHOT_PATH.iterdir()
    ])
else:
    print("Checkpoint is missing. Complete 8zeroshot first.")

Project root: /content/nlp-ku-project
OK /content/nlp-ku-project/src/challenge_eval.py
OK /content/nlp-ku-project/student_starter/validate_challenge.py
OK /content/nlp-ku-project/student_starter/challenge.json

Zero-shot checkpoint: /content/nlp-ku-project/checkpoints/zero_shot_telugu_qa
Checkpoint files: ['tokenizer_config.json', 'tokenizer.json', 'best_state_dict.pt', 'training_history.csv', 'config.json', 'model.safetensors']


In [36]:
import json

from student_starter.validate_challenge import validate_records

from src.challenge_eval import (
    apply_threshold,
    get_span_scores,
    qa_result_rows,
    summarise_challenge,
)

records = json.loads(
    CHALLENGE_PATH.read_text(encoding="utf-8")
)

validation_errors = validate_records(
    records,
    expected_per_member=10,
)

if validation_errors:
    raise ValueError(
        "Challenge validation failed:\n"
        + "\n".join(validation_errors)
    )

challenge_df = pd.DataFrame(records)

assert len(challenge_df) == 30, (
    f"Expected 30 records, found {len(challenge_df)}"
)

assert challenge_df["pair_id"].nunique() == 15, (
    "Expected 15 matched pairs."
)

print("Challenge set validated.")
print("Records:", len(challenge_df))
print("Pairs:", challenge_df["pair_id"].nunique())

print("\nLanguage distribution:")
display(challenge_df["lang"].value_counts().rename("records").to_frame())

print("\nColumns:")
print(challenge_df.columns.tolist())

Challenge set validated.
Records: 30
Pairs: 15

Language distribution:


,records
lang,
te,14
ar,8
ko,8



Columns:
['id', 'pair_id', 'created_by', 'question', 'context', 'lang', 'answerable', 'answer_start', 'answer', 'source_url']


In [37]:
assert ZERO_SHOT_PATH.exists(), (
    f"Missing checkpoint: {ZERO_SHOT_PATH}. "
    "Run the training and saving cells in 8zeroshot first."
)

assert (ZERO_SHOT_PATH / "config.json").exists(), (
    "The checkpoint folder has no config.json. "
    "Check that model.save_pretrained() saved to this directory."
)

zero_tokenizer = AutoTokenizer.from_pretrained(
    ZERO_SHOT_PATH,
    use_fast=True,
)

zero_model = AutoModelForQuestionAnswering.from_pretrained(
    ZERO_SHOT_PATH,
    attn_implementation="eager",
).to(device)

zero_model.eval()

print("Zero-shot model loaded successfully.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Zero-shot model loaded successfully.


In [38]:
ZERO_SHOT_THRESHOLD = 0.1306989336

print("Frozen zero-shot threshold:", ZERO_SHOT_THRESHOLD)

Frozen zero-shot threshold: 0.1306989336


In [39]:
langs = challenge_df["lang"].tolist()

zero_shot_scores = get_span_scores(
    zero_model,
    zero_tokenizer,
    challenge_df,
    device,
)

print("Number of score records:", len(zero_shot_scores))
print("Number of challenge records:", len(challenge_df))

assert len(zero_shot_scores) == len(challenge_df), (
    "The number of model scores does not match the challenge records."
)

Number of score records: 30
Number of challenge records: 30


In [40]:
zero_shot_predictions = apply_threshold(
    zero_shot_scores,
    langs,
    ZERO_SHOT_THRESHOLD,
)

zero_shot_results = qa_result_rows(
    challenge_df,
    zero_shot_predictions,
)

print("Zero-shot challenge evaluation completed.")
display(zero_shot_results.head())

Zero-shot challenge evaluation completed.


,id,pair_id,lang,answerable,prediction,em,f1,pred_answerable
0,rapa-nui-answerable,rapa-nui,ar,True,Easter Island,0.0,0.000000,True
1,rapa-nui-unanswerable,rapa-nui,ar,False,Rapa Nui,0.0,0.000000,True
2,chinchorro-answerable,chinchorro,ko,True,Chinchorro culture,0.0,0.666667,True
3,chinchorro-unanswerable,chinchorro,ko,False,Chinchorro culture,0.0,0.000000,True
4,alma-answerable,alma,te,True,16 kilometres,1.0,1.000000,True


In [41]:
print(zero_shot_results.columns.tolist())

['id', 'pair_id', 'lang', 'answerable', 'prediction', 'em', 'f1', 'pred_answerable']


In [42]:
my_results = {
    "Zero-shot Arabic/Korean QA": zero_shot_results
}

zero_shot_summary = summarise_challenge(my_results)

zero_shot_by_language = summarise_challenge(
    my_results,
    by_language=True,
)

print("OVERALL ZERO-SHOT CHALLENGE RESULTS")
display(zero_shot_summary)

print("ZERO-SHOT RESULTS BY LANGUAGE")
display(zero_shot_by_language)

OVERALL ZERO-SHOT CHALLENGE RESULTS


,model,n_answerable,answered,n_unanswerable,abstained,em,f1,n_pairs,pairs_correct,pairs_exact
0,Zero-shot Arabic/Korean QA,15.0,1.0,15.0,0.0,0.5,0.679,15.0,0.0,0.0


ZERO-SHOT RESULTS BY LANGUAGE


,model,lang,n_answerable,answered,n_unanswerable,abstained,em,f1,n_pairs,pairs_correct,pairs_exact
0,Zero-shot Arabic/Korean QA,ar,4,1.0,4,0.0,0.0,0.371,4,0.0,0.0
1,Zero-shot Arabic/Korean QA,ko,4,1.0,4,0.0,0.5,0.667,4,0.0,0.0
2,Zero-shot Arabic/Korean QA,te,7,1.0,7,0.0,1.0,1.000,7,0.0,0.0


In [52]:
print("\nTelugu challenge records:")
telugu_records = challenge_df[
    challenge_df["lang"] == "te"
].sort_values("pair_id")

display(telugu_records)


Telugu challenge records:


,id,pair_id,created_by,question,context,lang,answerable,answer_start,answer,source_url
4,alma-answerable,alma,Guillermo Martinez,ALMA యాంటెన్నాల మధ్య గరిష్ఠ దూరం ఎంత?,"ALMA comprises 66 high-precision antennas, spr...",te,True,74,16 kilometres,https://www.eso.org/public/chile/teles-instr/a...
5,alma-unanswerable,alma,Guillermo Martinez,ALMA యాంటెన్నాల మధ్య కనిష్ఠ దూరం ఎంత?,"ALMA comprises 66 high-precision antennas, spr...",te,False,-1,,https://www.eso.org/public/chile/teles-instr/a...
26,capital_answerable,capital,Zarin Raihana,ఫ్రాన్స్ రాజధాని ఏది?,Paris is the capital and largest city of France.,te,True,0,Paris,https://en.wikipedia.org/wiki/Paris
27,capital_unanswerable,capital,Zarin Raihana,ఇటలీ రాజధాని ఏది?,Paris is the capital and largest city of France.,te,False,-1,,https://en.wikipedia.org/wiki/Paris
24,distance_answerable,distance,Zarin Raihana,భూమి నుండి సూర్యుడు ఎంత దూరంలో ఉన్నాడు?,The Sun is about 150 million kilometres from E...,te,True,17,150 million kilometres,https://en.wikipedia.org/wiki/Sun
25,distance_unanswerable,distance,Zarin Raihana,భూమి నుండి చంద్రుడు ఎంత దూరంలో ఉన్నాడు?,The Sun is about 150 million kilometres from E...,te,False,-1,,https://en.wikipedia.org/wiki/Sun
18,great-barrier-reef-answerable,great-barrier-reef,Amna Sohaib,గ్రేట్ బ్యారియర్ రీఫ్ ఏ సంవత్సరంలో ప్రపంచ వారస...,The Great Barrier Reef can be seen from outer ...,te,True,303,1981,https://en.wikipedia.org/wiki/Great_Barrier_Reef
19,great-barrier-reef-unanswerable,great-barrier-reef,Amna Sohaib,గ్రేట్ బ్యారియర్ రీఫ్‌ను ఏ సంవత్సరంలో ప్రపంచ వ...,The Great Barrier Reef can be seen from outer ...,te,False,-1,,https://en.wikipedia.org/wiki/Great_Barrier_Reef
28,location-answerable,location,Zarin Raihana,ఈఫిల్ టవర్ ఏ నగరంలో ఉంది?,"The Eiffel Tower is located in Paris, France.",te,True,31,Paris,https://en.wikipedia.org/wiki/Eiffel_Tower
29,location-unanswerable,location,Zarin Raihana,లండన్ ఐ ఏ నగరంలో ఉంది?,"The Eiffel Tower is located in Paris, France.",te,False,-1,,https://en.wikipedia.org/wiki/Eiffel_Tower


In [49]:
if "created_by" in challenge_df.columns:
    print("Contributors:", challenge_df["created_by"].unique())

    MY_CONTRIBUTOR_ID = "Zarin Raihana"

    my_challenge_records = challenge_df[
        challenge_df["created_by"] == MY_CONTRIBUTOR_ID
    ].sort_values("pair_id")

else:
    # Fallback: inspect all Telugu records.
    my_challenge_records = telugu_records.copy()

Contributors: <ArrowStringArray>
['Guillermo Martinez', 'Amna Sohaib', 'Zarin Raihana']
Length: 3, dtype: str


In [53]:
print("Prediction columns:")
print(zero_shot_results.columns.tolist())

if "pair_id" in zero_shot_results.columns:
    my_pair_ids = my_challenge_records["pair_id"].unique()

    my_pair_results = zero_shot_results[
        zero_shot_results["pair_id"].isin(my_pair_ids)
    ].sort_values("pair_id")

    display(my_pair_results)

Prediction columns:
['id', 'pair_id', 'lang', 'answerable', 'prediction', 'em', 'f1', 'pred_answerable']


,id,pair_id,lang,answerable,prediction,em,f1,pred_answerable
26,capital_answerable,capital,te,True,Paris,1.0,1.0,True
27,capital_unanswerable,capital,te,False,Paris,0.0,0.0,True
24,distance_answerable,distance,te,True,150 million kilometres,1.0,1.0,True
25,distance_unanswerable,distance,te,False,150 million kilometres,0.0,0.0,True
28,location-answerable,location,te,True,Paris,1.0,1.0,True
29,location-unanswerable,location,te,False,"Paris, France",0.0,0.0,True
20,neil-armstrong-answerable,neil-armstrong,te,True,Neil Armstrong,1.0,1.0,True
21,neil-armstrong-unanswerable,neil-armstrong,te,False,Neil Armstrong,0.0,0.0,True
22,space-shuttle-answerable,space-shuttle,te,True,Space Shuttle Discovery,1.0,1.0,True
23,space-shuttle-unanswerable,space-shuttle,te,False,Space Shuttle Discovery,0.0,0.0,True


In [56]:
# my_challenge_records and zero_shot_results
my_predictions = my_challenge_records.merge(
    zero_shot_results[
        ["id", "prediction", "em", "f1", "pred_answerable"]
    ],
    on="id",
    how="left",
)

pair_rows = []

for pair_id, group in my_predictions.groupby("pair_id"):
    answerable = group[group["answerable"] == True]
    unanswerable = group[group["answerable"] == False]

    if len(answerable) != 1 or len(unanswerable) != 1:
        raise ValueError(f"Unexpected pair structure: {pair_id}")

    a = answerable.iloc[0]
    u = unanswerable.iloc[0]

    answer_correct = bool(a["em"] == 1.0)
    abstained_correctly = not bool(u["pred_answerable"])

    pair_rows.append({
        "pair_id": pair_id,
        "answerable_question": a["question"],
        "gold_answer": a["answer"],
        "predicted_answer": a["prediction"],
        "answer_em": a["em"],
        "answer_f1": a["f1"],
        "unanswerable_question": u["question"],
        "unanswerable_prediction": u["prediction"],
        "abstained_correctly": abstained_correctly,
        "pair_correct": answer_correct and abstained_correctly,
    })

my_pair_results = pd.DataFrame(pair_rows)

display(my_pair_results)

my_pair_results.to_csv(
    RESULTS_DIR / "my_challenge_pair_analysis.csv",
    index=False,
)

print(
    "Fully correct pairs:",
    int(my_pair_results["pair_correct"].sum()),
    "/",
    len(my_pair_results),
)

,pair_id,answerable_question,gold_answer,predicted_answer,answer_em,answer_f1,unanswerable_question,unanswerable_prediction,abstained_correctly,pair_correct
0,capital,ఫ్రాన్స్ రాజధాని ఏది?,Paris,Paris,1.0,1.0,ఇటలీ రాజధాని ఏది?,Paris,False,False
1,distance,భూమి నుండి సూర్యుడు ఎంత దూరంలో ఉన్నాడు?,150 million kilometres,150 million kilometres,1.0,1.0,భూమి నుండి చంద్రుడు ఎంత దూరంలో ఉన్నాడు?,150 million kilometres,False,False
2,location,ఈఫిల్ టవర్ ఏ నగరంలో ఉంది?,Paris,Paris,1.0,1.0,లండన్ ఐ ఏ నగరంలో ఉంది?,"Paris, France",False,False
3,neil-armstrong,చంద్రుడిపై నడిచిన మొదటి వ్యక్తి ఎవరు?,Neil Armstrong,Neil Armstrong,1.0,1.0,అంగారక గ్రహంపై నడిచిన మొదటి వ్యక్తి ఎవరు?,Neil Armstrong,False,False
4,space-shuttle,హబుల్ అంతరిక్ష దూరదర్శినిని ఏ స్పేస్ షటిల్ తీస...,Space Shuttle Discovery,Space Shuttle Discovery,1.0,1.0,జేమ్స్ వెబ్ అంతరిక్ష దూరదర్శినిని ఏ స్పేస్ షటి...,Space Shuttle Discovery,False,False


Fully correct pairs: 0 / 5


In [51]:
RESULTS_DIR = PROJECT_ROOT / "results" / "zero_shot"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

zero_shot_results.to_csv(
    RESULTS_DIR / "zero_shot_challenge_predictions.csv",
    index=False,
)

zero_shot_summary.to_csv(
    RESULTS_DIR / "zero_shot_challenge_summary.csv",
    index=False,
)

zero_shot_by_language.to_csv(
    RESULTS_DIR / "zero_shot_challenge_by_language.csv",
    index=False,
)

my_challenge_records.to_csv(
    RESULTS_DIR / "my_challenge_records.csv",
    index=False,
)

print("Saved files:")
for path in sorted(RESULTS_DIR.glob("*.csv")):
    print(path)

Saved files:
/content/nlp-ku-project/results/zero_shot/my_challenge_records.csv
/content/nlp-ku-project/results/zero_shot/zero_shot_challenge_by_language.csv
/content/nlp-ku-project/results/zero_shot/zero_shot_challenge_predictions.csv
/content/nlp-ku-project/results/zero_shot/zero_shot_challenge_summary.csv


**Zero-Shot Model: Weeks 43–44 Challenge Evaluation**

The zero-shot Arabic/Korean QA model was evaluated on the shared challenge set, consisting of 30 records arranged into 15 matched answerable/unanswerable pairs across Arabic, Korean, and Telugu. The model checkpoint and no-answer threshold were fixed before challenge evaluation.

In my five Telugu pairs, the model predicted the correct answer span for all five answerable questions. However, it also produced non-empty answers for all five unanswerable counterparts. Therefore, none of my five pairs was fully correct under the pair-level evaluation criterion.

This result illustrates that extracting an answer from a context and recognizing when the context does not support an answer are separate challenges. The model performed well on answer extraction for these five answerable examples but failed to abstain when the question was changed to request information absent from the context.